# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading, reviewing, and exploring a FAIR dataset defined by a Croissant schema using the `mlcroissant` library. All dataset elements (record sets, fields, columns, etc.) are referenced by their unique `@id` fields, ensuring unambiguous entity selection.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields.

We will use the `@id` fields for all references. Let's list the available record sets, their `@id`s, and a sample of their contained fields.

In [ ]:
# List all record sets and their @id fields
record_sets = list(dataset.record_sets())
if not record_sets:
    print("No record sets detected in this dataset\n.")
else:
    print("Available Record Sets and their @id values:")
    for rs in record_sets:
        print(f"- {rs['@id']}: {rs.get('name', '[No name]')}")

# Display fields/columns within each record set by @id
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field.get('@id', '[No id]')
        field_name = field.get('name', '[No name]')
        print(f"  - Field @id: {field_id} | Name: {field_name}")

## 3. Data Extraction
Load data from chosen record sets using their `@id`s. The `mlcroissant` API always references entities by their `@id`. 

We will load all available record sets found in the previous overview into separate pandas DataFrames.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}.")
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

# Print available columns for the first (if any)
if dataframes:
    first_rs = record_set_ids[0]
    print(f"\nFields (@id as columns) in record set '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No dataframes available.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, or grouping. All selections use field/column `@id` values as found in the DataFrames.

In [ ]:
# Select a record set and numeric field by @id (update IDs if more are available)
if dataframes:
    # Use the first available record set as example
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        numeric_field_id = numeric_fields[0]  # Use first numeric field (by @id)
        print(f"Using numeric field: '{numeric_field_id}' in record set: '{record_set_id}'")

        # Filter rows where value > median
        thresh = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > thresh].copy()
        print(f"Filtered rows with {numeric_field_id} > {thresh}")
        display(filtered_df.head())

        # Normalize numeric field (z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized ({norm_col}):")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a categorical field
        non_numeric_fields = [col for col in df.columns if col not in numeric_fields]
        if non_numeric_fields:
            group_field = non_numeric_fields[0]
            print(f"\nGrouping by '{group_field}' field:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            display(grouped_df)
        else:
            print("No categorical field available for grouping.")
else:
    print("No DataFrame to perform EDA on.")

## 5. Visualization
Visualize the numeric field's distribution and its normalized counterpart, referring to columns by their `@id` values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_fields:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)

    plt.subplot(1,2,2)
    sns.histplot(filtered_df[norm_col].dropna(), kde=True)
    plt.title(f"Normalized {numeric_field_id} (filtered)")
    plt.xlabel(norm_col)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
This notebook demonstrated end-to-end usage of the `mlcroissant` library to load, explore, and process a dataset defined via a Croissant schema. All entities and columns were referenced by their `@id` identifiers. The sample EDA provided filtering, normalization, grouping, and visualization using the dataset's structure as discovered from the schema. For further exploration, modify the record set and field `@id`s to suit your analysis objectives.